# Comparing R scripts

This notebook adapts the two approaches in the original project. It uses small, synthetic R scripts so the example can be shared. Open the notebook from this project or its `notebooks` directory.

In [ ]:
from pathlib import Path
import sys

project = Path.cwd()
if project.name == "notebooks":
    project = project.parent
sys.path.insert(0, str(project / "src"))

from code_similarity import (
    calculate_structural_similarity,
    compare_files,
    get_r_files_in_directory,
    read_file,
)

files = get_r_files_in_directory(project / "examples")
assert len(files) == 3

## Compare file text

The original notebook included a text-based comparison. Here, I use the script's `SequenceMatcher` ratio to show how similar character sequences can give a high score.

In [ ]:
from itertools import combinations

for left, right in combinations(files, 2):
    score = calculate_structural_similarity(read_file(left), read_file(right))
    print(f"{Path(left).name} / {Path(right).name}: {score:.3f}")

## Compare BERT embeddings

I take the final-layer [CLS] vector for each file and calculate cosine similarity for each pair. The first run downloads `bert-base-uncased`. Both scores describe similarity and need context when interpreting a pair.

In [ ]:
scores = compare_files(files)
assert len(scores) == 3
for left, right, cosine, text_match in sorted(scores, key=lambda row: row[2:], reverse=True):
    print(f"{Path(left).name} / {Path(right).name}: cosine={cosine:.3f}, text={text_match:.3f}")